# 07 - Custom Models

This notebook walks through the OCI Vision custom model lifecycle:
creating a project, uploading training data, training a model, deploying
it, and running inference. This is a conceptual walkthrough that requires
real OCI credentials -- it cannot run in demo mode.

## Setup

In [ ]:
# !pip install oci-vision-ai[notebooks]

from oci_vision.core.client import VisionClient

# Demo mode for illustration; custom models require live credentials
client = VisionClient(demo=True)
print(f"Client ready (demo={client.is_demo})")
print()
print("NOTE: Custom model operations require real OCI credentials.")
print("This notebook is educational -- it shows the workflow and code")
print("patterns without executing live API calls.")

## The custom model lifecycle

OCI Vision custom models follow these steps:

1. **Create a project** -- organizational container in OCI Vision
2. **Prepare training data** -- labeled images in Object Storage
3. **Create a dataset** -- register the data with OCI Data Labeling
4. **Train the model** -- start a training job
5. **Deploy** -- create an endpoint for inference
6. **Predict** -- call the deployed model via VisionClient

In [ ]:
# Step 1: Create a project (conceptual -- requires OCI SDK)
#
# import oci
# config = oci.config.from_file()
# vision_client = oci.ai_vision.AIServiceVisionClient(config)
#
# project_details = oci.ai_vision.models.CreateProjectDetails(
#     display_name="my-custom-classifier",
#     compartment_id="ocid1.compartment.oc1..xxxxx",
#     description="Custom classifier for product categories"
# )
#
# project = vision_client.create_project(project_details).data
# print(f"Project ID: {project.id}")

print("Step 1: Create a project")
print("  -> vision_client.create_project(project_details)")
print("  -> Returns project OCID for subsequent steps")

In [ ]:
# Step 2: Prepare training data
#
# Training data layout in Object Storage:
# oci://my-bucket/training-data/
#   class_a/
#     img001.jpg
#     img002.jpg
#   class_b/
#     img003.jpg
#     img004.jpg
#   manifest.json  <-- optional metadata file

print("Step 2: Prepare training data")
print("  -> Upload labeled images to OCI Object Storage")
print("  -> Organize by class in subdirectories")
print("  -> Minimum 10 images per class (50+ recommended)")
print()
print("Supported formats: JPEG, PNG, BMP, TIFF")
print("Recommended: 224x224 to 1024x1024 pixels")

In [ ]:
# Step 3: Create a model (training job)
#
# model_details = oci.ai_vision.models.CreateModelDetails(
#     display_name="product-classifier-v1",
#     compartment_id="ocid1.compartment.oc1..xxxxx",
#     project_id=project.id,
#     model_type="IMAGE_CLASSIFICATION",  # or OBJECT_DETECTION
#     training_dataset=oci.ai_vision.models.ObjectStorageDataset(
#         dataset_type="OBJECT_STORAGE",
#         namespace_name="my-namespace",
#         bucket_name="my-bucket",
#         object_name="training-data/manifest.json"
#     ),
#     max_training_duration_in_hours=2.0,
#     is_quick_mode=True  # faster training, good for prototyping
# )
#
# model = vision_client.create_model(model_details).data
# print(f"Training started: {model.id}")

print("Step 3: Create and train the model")
print("  -> vision_client.create_model(model_details)")
print("  -> Model types: IMAGE_CLASSIFICATION or OBJECT_DETECTION")
print("  -> Quick mode: ~15-30 min; Full mode: 1-24 hours")
print("  -> Training is asynchronous -- poll for ACTIVE status")

In [ ]:
# Step 4: Check training status and metrics
#
# model = vision_client.get_model(model_id).data
# print(f"Status: {model.lifecycle_state}")
# print(f"Precision: {model.metrics.precision}")
# print(f"Recall: {model.metrics.recall}")
# print(f"F1 Score: {model.metrics.f1_score}")

# Simulated training metrics
print("Step 4: Training metrics (example)")
print()
metrics = {
    "Status": "ACTIVE",
    "Precision": 0.943,
    "Recall": 0.921,
    "F1 Score": 0.932,
    "Training Time": "27 minutes",
    "Total Images": 500,
    "Classes": 5,
}
for key, val in metrics.items():
    print(f"  {key:20s}: {val}")

## Explore the results

### Calling a custom model with VisionClient

Once a custom model is trained and active, you call it the same way
as the pretrained models -- just pass the model OCID.

In [ ]:
# Step 5: Run inference with a custom model
#
# client = VisionClient(compartment_id="ocid1.compartment.oc1..xxxxx")
#
# For classification:
# result = client.classify("new_product_image.jpg")
# print(result.labels)  # returns your custom class labels
#
# For the full analyze() pipeline:
# report = client.analyze("new_image.jpg", features=["classification"])

print("Step 5: Run inference")
print("  -> Same VisionClient API as pretrained models")
print("  -> client.classify('image.jpg') returns ClassificationResult")
print("  -> Labels match your custom training classes")
print()

# Simulated custom model result
from oci_vision.core.models import ClassificationResult, ClassificationLabel

custom_result = ClassificationResult(
    model_version="custom-v1",
    labels=[
        ClassificationLabel(name="Electronics", confidence=0.9534),
        ClassificationLabel(name="Furniture", confidence=0.0234),
        ClassificationLabel(name="Clothing", confidence=0.0156),
        ClassificationLabel(name="Food", confidence=0.0048),
        ClassificationLabel(name="Books", confidence=0.0028),
    ]
)

print("Simulated custom model output:")
for label in custom_result.labels:
    bar = '#' * int(label.confidence * 40)
    print(f"  {label.name:15s} {label.confidence_pct:6.2f}% {bar}")

## Visualize

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Custom model prediction distribution
names = [l.name for l in custom_result.labels]
confs = [l.confidence_pct for l in custom_result.labels]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#E63946' if c > 50 else '#A8DADC' for c in confs]
ax.barh(names, confs, color=colors)
ax.set_xlabel('Confidence (%)')
ax.set_title('Custom Model Prediction (Simulated)')
ax.set_xlim(0, 105)
for i, (name, conf) in enumerate(zip(names, confs)):
    ax.text(conf + 0.5, i, f'{conf:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('custom_model_prediction.png', dpi=100, bbox_inches='tight')
plt.show()

## Under the hood

In [ ]:
import json

# Custom model lifecycle summary
lifecycle = {
    "project": {
        "display_name": "my-custom-classifier",
        "id": "ocid1.aivisionproject.oc1..example"
    },
    "model": {
        "display_name": "product-classifier-v1",
        "model_type": "IMAGE_CLASSIFICATION",
        "lifecycle_state": "ACTIVE",
        "training_dataset": {
            "type": "OBJECT_STORAGE",
            "bucket": "my-bucket",
            "prefix": "training-data/"
        },
        "metrics": {
            "precision": 0.943,
            "recall": 0.921,
            "f1_score": 0.932
        }
    },
    "prediction": custom_result.model_dump()
}
print(json.dumps(lifecycle, indent=2, default=str))

## Try it yourself

1. **Create a project**: Use the OCI Console or SDK to create a
   Vision AI project in your compartment.
2. **Collect training data**: Gather 50+ images per class and upload
   them to Object Storage in labeled subdirectories.
3. **Quick mode first**: Start with `is_quick_mode=True` to get a
   baseline model in ~30 minutes, then train a full model.
4. **Compare models**: Train multiple versions with different data
   augmentation and compare metrics.
5. **A/B testing**: Deploy two model versions and compare predictions
   on the same test set.